# Quench-detector project summary

End-to-end walkthrough of everything built so far, in the order it was built. Each
section explains what the piece is for, runs it, and looks at the result -- this is
meant to be read top to bottom as the project map, not just a script dump.

**Layers, roughly in dependency order:**

1. **REBCO analytic physics** (`physics.py`, `thermal.py`) -- the ground-truth solver, from
   Marchevsky & Prestemon 2024.
2. **Bi-2223 analytic physics** (`physics_bi2223.py`, `bi2223_rollout.py`) -- a second,
   separate conductor model from the same paper's experimental section.
3. **The Gym env** (`env.py`) and rule-based **controllers** (`controllers.py`) built on
   top of (1).
4. **Datasets + sequence surrogate** (`datagen.py`, `rollout.py`, `seq_data.py`,
   `seq_surrogates.py`) -- a GRU trained to infer I*/margin from telemetry alone.
5. **FEM ground truth** (`fem_data.py`, `fem_env.py`) -- real spatially-resolved sensor
   data (RTDs + voltage taps) from a 3D electrothermal solve, brought in as both a
   tabular dataset and a drop-in replay env.
6. **Generalization checks** -- running the REBCO-trained surrogate against FEM and
   Bi-2223 trajectories it never saw.
7. **Spatial diagnostics** -- what the FEM data shows that no lumped model can.
8. **A real bug, found by writing tests** -- and what changed because of it.

See `docs/PROGRESS.md` for the dated log this notebook is summarizing.

In [ ]:
import sys, pathlib
repo_root = pathlib.Path.cwd()
if not (repo_root / "hts_tape_env").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import os
for v in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(v, "1")

import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

FEM_AVAILABLE = pathlib.Path("/eagle/fanc_ldrd/magnets_fem/sweep.h5").exists()
print("repo root:", repo_root)
print("FEM sweep reachable:", FEM_AVAILABLE)

## 1. REBCO analytic physics (`physics.py`)

The source of truth: a heat-balance solver for a single HTS tape conductor (REBCO,
Tc=85 K, Ic0=500 A by default), with an optional Cu/SS stabilizer that current can
redistribute into. `solve_runaway(T0, lambda)` finds the runaway threshold (I*, T*) by
tracing the equilibrium I(T) curve and locating its interior maximum.

In [ ]:
from hts_tape_env import physics

T0_grid = np.linspace(5, 80, 40)
Istar_stab = [physics.solve_runaway(T0, 20.0, mode="stab", lam_override=80.0)[0] for T0 in T0_grid]
Istar_no_stab = [physics.solve_runaway(T0, 20.0, mode="no_stab", lam_override=80.0)[0] for T0 in T0_grid]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(T0_grid, Istar_stab, label="mode='stab' (Cu/SS current sharing)")
ax.plot(T0_grid, Istar_no_stab, label="mode='no_stab'")
ax.set_xlabel("T0 [K]"); ax.set_ylabel("I* [A]")
ax.set_title("REBCO runaway current I*(T0), lambda=80 W/m/K")
ax.legend()
plt.show()
print("Stabilizer raises I* everywhere, as expected -- it gives current somewhere to go.")

## 2. Bi-2223 analytic physics (`physics_bi2223.py`)

A **separate** module, not a mode flag on `physics.py` -- see its docstring for why. Same
paper (Marchevsky & Prestemon 2024), but its Sec. 3 validates against an experimentally
measured Bi-2223 tape: measured I_c(T)/n(T)/eta plugged into the *no-stabilizer* heat
balance, no current-redistribution model. The paper doesn't publish a digitized I_c(T)/n(T)
table (Fig 6(e) is a scatter plot) -- the anchors here are read off that figure by eye and
flagged loudly in the module docstring. Only trustworthy up to ~10-20%, and only
well-behaved for T0 up to ~80-81 K (see the guard in `solve_runaway`).

In [ ]:
from hts_tape_env import physics_bi2223 as bi2223

T0_bi = np.linspace(bi2223.T_ANCHOR_K, 80.5, 30)
Istar_bi = [bi2223.solve_runaway(T0)[0] for T0 in T0_bi]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(T0_grid, Istar_stab, color="tab:blue")
axes[0].set_title("REBCO (mode='stab')"); axes[0].set_xlabel("T0 [K]"); axes[0].set_ylabel("I* [A]")
axes[1].plot(T0_bi, Istar_bi, color="tab:orange")
axes[1].set_title("Bi-2223 (measured Ic(T)/n(T))"); axes[1].set_xlabel("T0 [K]"); axes[1].set_ylabel("I* [A]")
fig.suptitle("Two separate conductors, two separate validated ranges (note the x-axis scales)")
fig.tight_layout(); plt.show()

Istar_check, Tstar_check = bi2223.solve_runaway(bi2223.T_ANCHOR_K)
print(f"Cross-check vs the paper's own reported ~260 A runaway at T0={bi2223.T_ANCHOR_K} K:")
print(f"  this reconstruction gives I*={Istar_check:.1f} A ({(Istar_check/260-1)*100:+.1f}%)")

## 3. The Gym env (`env.py`) and rule-based controllers (`controllers.py`)

`QuenchDetectorEnv` wraps `physics.py` into a Gymnasium MDP: the agent observes
`[T0, I, V, T_surface]` and chooses `continue` or `dump` at each ramp step. Reward
depends on whether a dump landed in the "danger band" before I* (a hit), too early
(false alarm), or never came (a miss). `controllers.py`'s `oracle_controller` dumps on
the ground-truth margin -- the reward ceiling any controller sharing this action space
could reach.

In [ ]:
from hts_tape_env.env import QuenchDetectorEnv
from hts_tape_env.controllers import oracle_controller, evaluate

env = QuenchDetectorEnv()
obs, info = env.reset(seed=0)
trace = [info]
terminated = truncated = False
while not (terminated or truncated):
    obs, reward, terminated, truncated, info = env.step(0)  # always continue
    trace.append(info)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot([t["margin"] for t in trace])
ax.axhline(0, color="gray", ls="--", lw=1)
ax.set_xlabel("step"); ax.set_ylabel("margin"); ax.set_title(f"One always-continue episode (T0={trace[0]['T0']:.1f} K)")
plt.show()

oracle_perf = evaluate(env, oracle_controller(env.dump_threshold), n_episodes=200, seed0=999)
print("Oracle controller over 200 episodes:", oracle_perf)

## 4. Sequence surrogate (`seq_surrogates.py`)

A GRU trained on `QuenchDetectorEnv` rollouts to predict the running margin and I* from
telemetry *alone* -- it never sees lambda, only how V and T_surface rise as I ramps. This
is the realistic, deployable version: `SequenceSurrogateAdapter` wires it into
`env.set_surrogate()` for streaming inference. Loading the existing checkpoint rather
than retraining (see `scripts/train_seq_surrogate.py` for the training run itself).

In [ ]:
from hts_tape_env.seq_surrogates import load_seq_surrogate, predict_runaway_surface

CKPT = repo_root / "models" / "seq_gru.pt"
result = load_seq_surrogate(CKPT)
print(f"Loaded {CKPT}")
print("Held-out metrics at train time:", result.metrics)

T0_grid_s = np.linspace(8.0, 72.0, 14)
lam_grid_s = np.linspace(24.0, 190.0, 14)
surfaces = predict_runaway_surface(result, T0_grid_s, lam_grid_s, seed=1000)
rel_err = np.abs(surfaces["Istar_pred"] - surfaces["Istar_analytic"]) / np.maximum(surfaces["Istar_analytic"], 1e-6)
print(f"I* surface median rel err (REBCO, home turf): {np.nanmedian(rel_err):.2%}")

TT, LL = np.meshgrid(T0_grid_s, lam_grid_s)
fig = plt.figure(figsize=(11, 4.5))
for i, (Z, title) in enumerate([(surfaces["Istar_analytic"], "Analytic I*"),
                                  (surfaces["Istar_pred"], "Surrogate I*\n(lambda inferred, not given)")]):
    ax = fig.add_subplot(1, 2, i + 1, projection="3d")
    ax.plot_surface(TT, LL, Z, cmap="turbo", edgecolor="k", linewidth=0.2)
    ax.set_xlabel("T0"); ax.set_ylabel("lambda"); ax.set_zlabel("I* [A]"); ax.set_title(title, fontsize=10)
fig.tight_layout(); plt.show()

## 5. FEM ground truth (`fem_data.py`)

Everything above is a *lumped* model: one (I, T) point for the whole tape. The FEM sweep
at `//eagle/fanc_ldrd/magnets_fem/sweep.h5` is real, spatially-resolved output from a 3D
electrothermal solve -- 6 runs at different current-sharing ratios (0.5 to 0.99), each
with 16 RTDs + 20 voltage taps at known positions along the tape. `load_fem_sweep`
reshapes it into a `rollout.py`-style table (same column names where a physical analog
exists; NaN, loudly documented, where one doesn't -- e.g. there's no FEM analog of I*
from a fixed-current-threshold sweep, since here it's the current-sharing *ratio* that's
swept, not transport current at fixed geometry).

In [ ]:
if FEM_AVAILABLE:
    from hts_tape_env.fem_data import load_fem_sweep, load_fem_positions
    fem_df = load_fem_sweep()
    fem_pos = load_fem_positions()
    summary = fem_df.groupby("episode_id").agg(
        ratio_label=("ratio_label", "first"), ratio=("ratio", "first"),
        status=("status", "first"), n_steps=("step", "count"),
        final_Tmax_K=("T_surface_K", "last"))
    display(summary)
else:
    print("FEM sweep not reachable from this environment -- skipping this section's execution "
          "(see docs/PROGRESS.md for results from a session where it was).")

## 6. FEM replay env (`fem_env.py`)

`FemReplayQuenchEnv` gives the FEM sweep the *same* `Discrete(2)` / `info["margin"]` /
`dump_threshold` contract as `QuenchDetectorEnv`, so `controllers.py` works against it
unmodified -- but the observation is the real 37-channel sensor grid (I0 + 16 RTDs + 20
V-taps, ordered by x-position), and `step()` just indexes into the precomputed FEM
trajectory (no PDE/boundary-condition solve at runtime). Only 6 fixed trajectories exist,
so treat this as a validation/spatial-prototyping env, not a large-scale RL training env.

In [ ]:
if FEM_AVAILABLE:
    from hts_tape_env.fem_env import FemReplayQuenchEnv

    fem_env = FemReplayQuenchEnv()
    print("observation shape:", fem_env.observation_space.shape, " n_episodes:", fem_env.n_episodes)

    obs, info = fem_env.reset_at("r0p99", seed=0)
    terminated = truncated = False
    while not (terminated or truncated):
        obs, reward, terminated, truncated, info = fem_env.step(0)  # never dump
    print(f"r0p99, never-dump: terminated={terminated}, runaway={info['runaway']}, "
          f"final margin={info['margin']:.3f}, reward={reward}")
else:
    print("skipped (FEM sweep not reachable)")

## 7. Generalization check: REBCO-trained surrogate vs FEM ground truth

The interesting question: does the surrogate trained purely on `physics.py`'s analytic
trajectories generalize to genuinely different (FEM) physics? Feeding FEM's
`(T0_K, I_true_A, voltage_true_V, T_surface_true_K)` straight into the same GRU
checkpoint via `seq_surrogates.predict_episode` -- no retraining.

In [ ]:
if FEM_AVAILABLE:
    from hts_tape_env.seq_data import TRUE_FEATURES
    from hts_tape_env.seq_surrogates import predict_episode

    fig, axes = plt.subplots(2, 3, figsize=(15, 7))
    fem_results = {}
    for ax, (label, g) in zip(axes.flat, fem_df.sort_values("step").groupby("ratio_label", sort=True)):
        X = g[TRUE_FEATURES].to_numpy(dtype=np.float32)
        r = predict_episode(result, X)
        fem_results[label] = r
        t = g["t_s"].to_numpy()
        ax2 = ax.twinx()
        ax.plot(t, r["margin_pred"], color="tab:blue")
        ax2.plot(t, r["breach_prob"], color="tab:red", alpha=0.6)
        ax2.axhline(0.5, color="tab:red", lw=0.5, ls="--")
        qs = g["quench_step"].iloc[0]
        if np.isfinite(qs):
            ax.axvline(t[int(qs)], color="black", ls=":")
        ax.set_title(f"{label} ({g['status'].iloc[0]})")
        ax.set_xlabel("t [s]"); ax.set_ylabel("margin_pred", color="tab:blue"); ax2.set_ylabel("breach_prob", color="tab:red")
    fig.suptitle("Sim-trained surrogate vs FEM ground truth (blue=margin_pred, red=breach_prob, dotted=FEM quench)")
    fig.tight_layout(); plt.show()

    for label in ("r0p95", "r0p99"):
        alarm = fem_results[label]["first_alarm_step"]
        print(f"{label}: breach_prob>0.5 {'never crosses' if alarm is None else f'crosses at step {alarm}'} "
              f"(FEM quench at step {fem_df[fem_df.ratio_label==label]['quench_step'].iloc[0]:.0f})")
else:
    print("skipped (FEM sweep not reachable)")

**Finding:** the surrogate's `breach_prob_pred` never crosses 0.5 on either real FEM
runaway (r0p95, r0p99) -- it misses both. On the 4 stable runs it correctly stays quiet
(no false alarms). A real sim-to-FEM generalization gap, not a bug in either model.

## 8. Generalization check: REBCO-trained surrogate vs Bi-2223 ground truth

Same idea, against `physics_bi2223.py`'s Bi-2223 trajectories (`bi2223_rollout.py`
generates them -- no Gym env needed there, see its docstring for why).

In [ ]:
from hts_tape_env.bi2223_rollout import generate_bi2223_grid_dataset

T0_grid_bi = np.linspace(77.16, 80.5, 12)
bi_df = generate_bi2223_grid_dataset(T0_grid_bi)

T0s, Ia, Ip = [], [], []
for episode_id, g in bi_df.sort_values("step").groupby("episode_id", sort=True):
    X = g[TRUE_FEATURES if FEM_AVAILABLE else ["T0_K", "I_true_A", "voltage_true_V", "T_surface_true_K"]].to_numpy(dtype=np.float32)
    r = predict_episode(result, X) if FEM_AVAILABLE else None
    if r is None:
        from hts_tape_env.seq_surrogates import predict_episode as _pe
        r = _pe(result, X)
    T0s.append(g["T0_K"].iloc[0]); Ia.append(g["Istar_A"].iloc[0]); Ip.append(r["istar_pred_A"])

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(T0s, Ia, "o-", label="analytic (physics_bi2223)", color="tab:orange")
ax.plot(T0s, Ip, "s--", label="REBCO-trained surrogate", color="tab:blue")
ax.set_xlabel("T0 [K]"); ax.set_ylabel("I* [A]"); ax.set_title("Bi-2223: analytic vs surrogate I*(T0)")
ax.legend(); plt.show()

print(f"analytic I* falls from {Ia[0]:.1f} to {Ia[-1]:.1f} A as T0 rises;",
      f"surrogate predicts {Ip[0]:.1f} to {Ip[-1]:.1f} A -- ",
      "same direction" if (Ip[-1]-Ip[0])*(Ia[-1]-Ia[0]) > 0 else "OPPOSITE direction")

**Finding:** the surrogate doesn't just fail to track the magnitude -- it gets the
*direction* wrong. Analytic I* falls smoothly as T0 rises (physically expected: less
margin left before Ic(T0) itself is exhausted); the surrogate's prediction drifts the
other way. Consistent with the FEM finding above: don't trust `margin_pred`/`I_star_pred`
outside REBCO's own training range (T0 in 5-75 K, this Ic0=500A/Tc=85K regime).

## 9. Spatial diagnostics: what the FEM data shows that no lumped model can

Every model above (REBCO, Bi-2223) collapses the whole tape to one (I, T) point. The FEM
sweep is real x-resolved data -- so, does the hot spot actually form uniformly, or
somewhere specific? Comparing a stable run (r0p5) against the runaway run (r0p99).

In [ ]:
if FEM_AVAILABLE:
    from hts_tape_env.fem_data import RTD_CHANNELS, VTAP_CHANNELS

    def channel_x(ratio_label, channels):
        ref = fem_pos[fem_pos["ratio_label"] == ratio_label].set_index("channel")
        return {c: float(ref.loc[c, "x_m"]) for c in channels if c in ref.index}

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    rtd1, rtd2 = [c for c in RTD_CHANNELS if c.startswith("RTD1_")], [c for c in RTD_CHANNELS if c.startswith("RTD2_")]
    for label, style in [("r0p5", "-"), ("r0p99", "--")]:
        g = fem_df[fem_df.ratio_label == label].sort_values("step")
        x_run = channel_x(label, RTD_CHANNELS)
        last = g.iloc[-1]
        for chans, marker in [(rtd1, "o"), (rtd2, "^")]:
            xs = np.array([x_run[c] for c in chans]); order = np.argsort(xs)
            ys = np.array([last[c] for c in chans])
            axes[0].plot(xs[order], ys[order], style, marker=marker, label=f"{label} {'RTD1' if marker=='o' else 'RTD2'}")
    axes[0].set_xlabel("x [m]"); axes[0].set_ylabel("T [K]"); axes[0].set_title("Final-step temperature vs position")
    axes[0].legend(fontsize=8)

    g99 = fem_df[fem_df.ratio_label == "r0p99"].sort_values("step")
    axes[1].plot(g99["t_s"], g99["TmaxLeft"], label="TmaxLeft", color="tab:blue")
    axes[1].plot(g99["t_s"], g99["TmaxRight"], label="TmaxRight", color="tab:red")
    axes[1].set_xlabel("t [s]"); axes[1].set_ylabel("T [K]"); axes[1].set_title("r0p99: left/right asymmetry")
    axes[1].legend()
    fig.tight_layout(); plt.show()

    hottest = max(RTD_CHANNELS, key=lambda c: g99.iloc[-1][c])
    print(f"Hottest RTD in the runaway run: {hottest} at x={channel_x('r0p99', RTD_CHANNELS)[hottest]:.4f} m "
          f"(T={g99.iloc[-1][hottest]:.1f} K) -- same location as the stable run's hottest point, just far less severe there.")
    print("Full 3-panel version (incl. per-segment voltage-tap localization): see scripts/plot_fem_spatial.py")
else:
    print("skipped (FEM sweep not reachable)")

**Finding:** the hot spot is sharply localized (a handful of cm on a 20 cm tape),
not spread across it -- confirming the paper's own point that REBCO's real behavior is
"far more spatially inhomogeneous" than a lumped model assumes. The local (per-segment)
voltage-tap reading right at that spot is the actual detection-relevant signal -- it
localizes the defect, not just flags "something's wrong somewhere" (see
`scripts/plot_fem_spatial.py` for that panel).

## 10. A bug, found by writing tests

Writing `tests/test_physics.py`'s `stable_surface_T` tests surfaced a real correctness
bug: the function's own docstring claimed `f(T0)` and `f(Tc)` always have opposite signs
(so any `(T0, Tc)` bracket would find the root), but that's wrong -- `Qin(I,T)` *diverges*
as `T -> Tc` (Ic(T) -> 0 in the denominator), not drops to 0. With same-signed endpoints,
`brentq` silently failed to bracket either root, and `env.py` fell back to reporting
`T_surface = T0` (no self-heating) -- **~40% of the time**, across a random
`(T0, lambda, I/I*)` sweep in the default `mode='stab'`. The exact same bracket bug
existed in `physics_bi2223.py` too.

**Fix:** both functions now accept a `T_hi` hint -- pass the operating point's own T*
(already computed by `solve_runaway`, free to reuse) as the upper search bound instead of
`Tc`/`T_x`. `env.py`, `datagen.py`, and `bi2223_rollout.py` were updated to pass it.

In [ ]:
# Before/after, on the same random (T0, lambda, I/I*) draws:
rng = np.random.default_rng(0)
n_bad_old, n_bad_new, n_total = 0, 0, 0
for _ in range(200):
    T0 = float(rng.uniform(5, 75)); lam = float(rng.uniform(20, 200))
    Istar, Tstar = physics.solve_runaway(T0, 20.0, mode="stab", lam_override=lam)
    if not np.isfinite(Istar):
        continue
    I = float(rng.uniform(0.5, 0.99)) * Istar
    T_old = physics.stable_surface_T(I, T0, 20.0, mode="stab", lam_override=lam)               # T_hi defaults to Tc
    T_new = physics.stable_surface_T(I, T0, 20.0, mode="stab", lam_override=lam, T_hi=Tstar)    # fixed
    n_total += 1
    n_bad_old += int(not np.isfinite(T_old))
    n_bad_new += int(not np.isfinite(T_new))

print(f"pre-fix (T_hi=Tc):  {n_bad_old}/{n_total} = {n_bad_old/n_total:.1%} of draws returned NaN")
print(f"post-fix (T_hi=T*): {n_bad_new}/{n_total} = {n_bad_new/n_total:.1%} of draws returned NaN")
print()
print("Caveat: models/seq_gru.pt and data/rollout_main.parquet used elsewhere in this notebook")
print("were generated AFTER this fix landed. If you have older artifacts from before this")
print("session, regenerate them -- their T_surface channel was silently wrong ~40% of the time.")

## 11. Test suite (`tests/`)

Added alongside this notebook: `pytest` coverage for every module above (physics, thermal,
env, controllers, physics_bi2223, bi2223_rollout, fem_data, fem_env), including the
regression test that pins down the bug fix in section 10. FEM-dependent tests skip
gracefully (not fail) when the `//eagle` mount isn't reachable.

```
.venv/bin/python3 -m pytest tests/ -v
```

In [ ]:
import subprocess
res = subprocess.run(["python3", "-m", "pytest", "tests/", "-q"], cwd=repo_root,
                      capture_output=True, text=True,
                      env={**os.environ, "OPENBLAS_NUM_THREADS": "1", "OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1"})
print(res.stdout[-1500:])
print(res.stderr[-500:] if res.returncode != 0 else "")

## Summary

- **REBCO and Bi-2223 are separate models on purpose** -- different current-sharing
  physics, different validated ranges, and merging them would blur a real distinction the
  reference paper itself draws.
- **The sequence surrogate is REBCO-specific.** It performs well (~2.8% median I* error)
  on its own training distribution, but both the FEM check and the Bi-2223 check show it
  does not generalize past that -- on Bi-2223 it gets the *direction* of I*(T0) wrong, and
  on FEM it misses both real runaway cases entirely. Don't condition a controller on
  `margin_pred` outside REBCO's own parameter range without addressing this first.
- **REBCO's real spatial behavior is far from uniform** -- the FEM hot spot is sharply
  localized, and a real voltage-tap array would actually localize it, which the lumped
  models structurally cannot represent.
- **Writing tests found a real ~40%-of-the-time bug**, now fixed with a regression test in
  place -- the practical case for the "add unit tests and checks" ask, not just process.

**Open threads**, roughly in order of what unblocks the most:
1. Train (or fine-tune) a surrogate on a wider mix -- Bi-2223 rollouts and/or FEM
   trajectories -- rather than REBCO analytic sweeps alone.
2. Re-digitize Fig 6(e) of the reference paper properly for exact Bi-2223 I_c(T)/n(T).
3. Train something against `fem_env.py`'s spatial (RTD-array) observation directly,
   instead of only running the lumped-trained surrogate on it.
4. More FEM sweep points, if/when available, to extend past the current 6 fixed runs.